# Reviewer 4: additional detector architecture

This notebook has **not been run** as part of the local revision. Choose a CUDA GPU runtime. It trains YOLO11m on 716 training trees, selects the checkpoint using 96 validation trees, then evaluates 141 test trees with the same Ridge+F0 counter used for YOLO26m. Images require access to the SawitMVC Hugging Face dataset. Architecture-specific defaults are saved; this is not a claim that every training detail is identical.

Upload `reviewer-gpu-bundle.zip` in the next cell. The bundle contains revision code, GT annotations and the reference detector cache; no credentials or images.

In [ ]:
%pip install -q ultralytics==8.4.52 scikit-learn==1.8.0 scipy pandas numpy huggingface_hub pyyaml


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import os, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.cuda.get_device_name(0))
uploaded = files.upload()
assert 'reviewer-gpu-bundle.zip' in uploaded
workspace = Path('/content/sawit-reviewer').resolve()
workspace.mkdir(exist_ok=True)
with ZipFile('reviewer-gpu-bundle.zip') as archive:
    for item in archive.infolist():
        target = (workspace / item.filename).resolve()
        assert target.is_relative_to(workspace), 'Unsafe archive member'
    archive.extractall(workspace)
os.chdir(workspace)


In [ ]:
from huggingface_hub import login, snapshot_download
login()  # authenticate in this notebook, never put a token in the manuscript
snapshot_download('ULM-DS-Lab/SawitMVC-YOLO', repo_type='dataset', local_dir='SawitMVC-YOLO', token=True)


In [ ]:
!python experiments/revision/a12_second_detector.py --prepare-only


## Train and evaluate

The next cell starts 60-epoch training (batch 32, imgsz 640, seed 42). Do not tune on test results. If interrupted after training, resume inference with `--weights runs/reviewer-yolo11/train/weights/best.pt --device 0`; cached trees are checked against the checkpoint hash.

In [ ]:
!python experiments/revision/a12_second_detector.py --device 0


In [ ]:
import json
from pprint import pprint
metrics = Path('results/revision/a12_second_detector_metrics.json')
assert metrics.exists(), 'Training/evaluation did not finish; inspect the output above'
pprint(json.loads(metrics.read_text()))


In [ ]:
from zipfile import ZIP_DEFLATED
result_zip = Path('/content/reviewer-yolo11-results.zip')
with ZipFile(result_zip, 'w', ZIP_DEFLATED) as archive:
    for directory in ['results/revision', 'predictions', 'runs/reviewer-yolo11']:
        for path in Path(directory).rglob('*'):
            if path.is_file() and 'y26mv2_per_tree' not in path.parts:
                archive.write(path, path.as_posix())
files.download(str(result_zip))


Return the output bundle to update Table VI, the scope discussion, and the response to Reviewer 4. Do not mark the second architecture request completed before the metrics and checkpoint provenance are available.

Model documentation: https://docs.ultralytics.com/models/yolo11/